# Programowanie w języku Python

## Wykład 9 — Błędy, wyjątki i diagnostyka programu

### dr inż. Waldemar Bauer

## Cel wykładu

Po dzisiejszych zajęciach powinieneś:

- rozróżniać błędy składniowe, wyjątki i błędy logiczne,
- umieć czytać traceback,
- obsługiwać konkretne wyjątki przez `try / except`,
- rozumieć `else` i `finally`,
- świadomie używać `raise`,
- tworzyć proste własne wyjątki,
- znać chaining wyjątków przez `raise ... from ...`,
- unikać `except:` bez typu,
- rozumieć podejścia EAFP i LBYL,
- korzystać z `breakpoint()`,
- używać modułu `logging` do diagnostyki programu.

## Program może nie działać na kilka sposobów

```text
1. BŁĄD SKŁADNI
   Python nie potrafi zbudować programu.

2. WYJĄTEK W CZASIE WYKONANIA
   składnia jest poprawna, ale operacja się nie powiedzie.

3. BŁĄD LOGICZNY
   program działa, ale daje zły wynik.
```

Każdy z tych problemów diagnozujemy inaczej.

## Błąd składniowy

Przykład:

```python
if temperature > 30
    print("ALARM")
```

Brakuje `:`.

Interpreter nie rozpocznie normalnego wykonywania programu, dopóki składnia nie będzie poprawna.

## Wyjątek

Kod:

```python
value = int("abc")
```

jest składniowo poprawny.

Dopiero podczas wykonywania Python odkrywa, że tekstu `"abc"` nie można przekonwertować na `int`.

In [ ]:
try:
    value = int("abc")
except ValueError as error:
    print(type(error).__name__)
    print(error)

## Błąd logiczny

Przykład:

```python
def celsius_to_kelvin(value):
    return value + 237.15
```

Kod działa i nie zgłasza wyjątku.

Ale wynik jest niepoprawny.

Wyjątki **nie zastępują testowania poprawności programu**.

## Traceback — ślad wykonania

Gdy wyjątek nie zostanie obsłużony, Python pokazuje traceback.

Czytaj go od końca:

```text
typ wyjątku + komunikat
↑
miejsce, w którym wyjątek powstał
↑
kolejne wywołania funkcji
```

Traceback jest informacją diagnostyczną, a nie tylko „czerwonym tekstem”.

## Przykład stosu wywołań

```python
def parse_value(text):
    return float(text)

def load_measurement(text):
    return parse_value(text)

load_measurement("brak")
```

Wyjątek powstaje w `float()`, ale traceback pokaże również drogę przez nasze funkcje.

## Podstawowa obsługa wyjątku

```python
try:
    ...
except ValueError:
    ...
```

Kod w `except` wykona się tylko wtedy, gdy w bloku `try` pojawi się pasujący wyjątek.

In [ ]:
text = "21.7"

try:
    value = float(text)
except ValueError:
    print("Niepoprawna liczba")
else:
    print("Wartość:", value)

## Łapiemy konkretny wyjątek

Zamiast:

```python
except:
    ...
```

preferujemy:

```python
except ValueError:
    ...
```

Różne awarie wymagają różnych reakcji.

## Dlaczego `except:` jest niebezpieczne?

Bare `except:` przechwytuje także wyjątki, których zwykle nie chcemy ukrywać.

Może zamaskować:

- błąd w kodzie,
- przerwanie programu,
- nieoczekiwany problem systemowy.

Jeśli potrzebujemy ogólnej granicy błędów, częściej używamy:

```python
except Exception as error:
    ...
```

ale również świadomie.

## Kilka rodzajów wyjątków

Przy pracy z danymi często spotkamy:

```text
ValueError
TypeError
KeyError
IndexError
FileNotFoundError
PermissionError
JSONDecodeError
ZeroDivisionError
```

Nie trzeba uczyć się pełnej listy na pamięć.

Ważne jest rozpoznawanie typu problemu.

## Kilka bloków `except`

Możemy reagować inaczej na różne problemy:

In [ ]:
from pathlib import Path

path = Path("data") / "measurement.txt"

try:
    text = path.read_text(encoding="utf-8")
    value = float(text)
except FileNotFoundError:
    print("Nie znaleziono pliku")
except ValueError:
    print("Plik nie zawiera poprawnej liczby")

## Kolejność ma znaczenie

Bardziej szczegółowe wyjątki umieszczamy przed bardziej ogólnymi.

```python
except FileNotFoundError:
    ...

except OSError:
    ...

except Exception:
    ...
```

Jeżeli najpierw przechwycimy szeroki typ, dalszy bardziej szczegółowy blok może nigdy nie zostać użyty.

## Obiekt wyjątku

```python
except ValueError as error:
```

wiąże nazwę `error` z obiektem wyjątku.

Możemy wykorzystać jego komunikat diagnostyczny.

In [ ]:
try:
    int("Python")
except ValueError as error:
    print("Typ:", type(error).__name__)
    print("Komunikat:", error)

## `else`

Blok `else` wykonuje się, jeżeli `try` zakończył się bez wyjątku.

```python
try:
    value = float(text)
except ValueError:
    ...
else:
    use(value)
```

Dzięki temu w `try` pozostaje tylko kod, który rzeczywiście może zgłosić oczekiwany wyjątek.

In [ ]:
text = "21.7"

try:
    value = float(text)
except ValueError:
    print("Niepoprawny pomiar")
else:
    print(f"Pomiar: {value:.1f}")

## `finally`

Blok `finally` wykona się niezależnie od tego, czy pojawił się wyjątek.

```python
try:
    ...
finally:
    ...
```

Przydaje się do sprzątania zasobów.

In [ ]:
try:
    print("Rozpoczynam operację")
    result = 10 / 2
finally:
    print("Kończę operację")

print(result)

## `with` jest często lepsze od ręcznego `finally`

Dla plików na W8 używaliśmy:

```python
with path.open(...) as file:
    ...
```

To właśnie mechanizm zarządzania zasobem.

Nie musimy ręcznie pisać:

```python
try:
    ...
finally:
    file.close()
```

## Nie przechwytuj wyjątku, którego nie potrafisz obsłużyć

Zły wzorzec:

```python
try:
    important_operation()
except Exception:
    pass
```

Program traci informację o problemie.

Obsługa wyjątku powinna mieć sensowną reakcję: naprawić sytuację, zgłosić ją, zmienić poziom abstrakcji albo zakończyć operację.

## `raise` — zgłaszanie wyjątku

Nasza funkcja również może wykryć niepoprawny stan:

In [ ]:
def validate_temperature(value):
    if value < -273.15:
        raise ValueError(
            "Temperatura nie może być niższa od zera absolutnego"
        )

    return value

print(validate_temperature(21.7))

## Wyjątek jako część kontraktu funkcji

Funkcja może informować:

```text
dla poprawnych danych
→ zwracam wynik

dla danych niedopuszczalnych
→ zgłaszam określony wyjątek
```

To często jest czytelniejsze niż specjalne wartości takie jak:

```python
return -999
```

## Własny typ wyjątku

Jeśli domena programu wymaga własnego rodzaju błędu:

In [ ]:
class MeasurementError(Exception):
    """Base error for invalid measurement data."""


def parse_measurement(text):
    try:
        return float(text)
    except ValueError as error:
        raise MeasurementError(
            f"Niepoprawny pomiar: {text!r}"
        ) from error

## Chaining: `raise ... from ...`

W przykładzie:

```python
raise MeasurementError(...) from error
```

zachowujemy informację:

```text
ValueError
↓ spowodował
MeasurementError
```

To pozwala zmienić poziom abstrakcji bez utraty przyczyny problemu.

In [ ]:
try:
    parse_measurement("brak")
except MeasurementError as error:
    print(error)
    print("Przyczyna:", repr(error.__cause__))

## Kiedy własny wyjątek ma sens?

Własny typ warto rozważyć, gdy kod wywołujący powinien móc odróżnić:

```text
błąd domeny aplikacji
```

od:

```text
ogólnego ValueError / OSError / ...
```

Nie tworzymy nowej klasy wyjątku tylko po to, aby zmienić tekst komunikatu.

## LBYL — Look Before You Leap

Jedno podejście:

```python
if key in data:
    value = data[key]
else:
    ...
```

Najpierw sprawdzamy warunek, potem wykonujemy operację.

## EAFP — Easier to Ask Forgiveness than Permission

Częsty Pythonowy styl:

```python
try:
    value = data[key]
except KeyError:
    ...
```

Próbujemy wykonać normalną operację i reagujemy na konkretną awarię.

## EAFP nie znaczy „łap wszystko”

Nie chodzi o:

```python
try:
    cały_program()
except:
    pass
```

EAFP działa dobrze, gdy:

- wiemy, jaki wyjątek może powstać,
- potrafimy sensownie na niego zareagować,
- blok `try` jest możliwie mały.

## Diagnostyka: najpierw odtwórz problem

Praktyczna procedura:

```text
1. Znajdź najmniejszy przypadek, który wywołuje błąd.
2. Przeczytaj typ wyjątku i traceback.
3. Sprawdź wartości wejściowe.
4. Ustal, w której funkcji powstaje problem.
5. Dopiero potem zmieniaj kod.
```

Losowe modyfikowanie programu zwykle wydłuża diagnozę.

## `repr()` jako narzędzie diagnostyczne

Dla tekstu:

```python
print(text)
```

może ukrywać białe znaki.

```python
print(repr(text))
```

pokazuje m.in. `\n`, `\t` i spacje.

In [ ]:
text = "21.7\n"

print(text)
print(repr(text))

## `breakpoint()`

Python ma wbudowany punkt zatrzymania:

```python
breakpoint()
```

Po dojściu do niego program uruchamia debugger i pozwala badać stan programu krok po kroku.

To lepsze narzędzie niż dodawanie dziesiątek przypadkowych `print()`.

## Minimalny przykład debugowania

```python
def calculate_average(values):
    total = sum(values)
    breakpoint()
    return total / len(values)
```

W debuggerze możemy sprawdzić:

```text
values
total
len(values)
```

oraz kontynuować wykonanie.

## `print()` czy logging?

`print()` jest dobry do:

- prostych wyników programu,
- krótkich eksperymentów,
- nauki.

W większym programie diagnostyka powinna zwykle korzystać z:

```python
logging
```

## Poziomy logowania

Standardowe poziomy:

```text
DEBUG
INFO
WARNING
ERROR
CRITICAL
```

Pozwalają rozdzielić szczegółową diagnostykę od informacji istotnych dla użytkownika lub administratora.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s",
)

logging.debug("Szczegóły diagnostyczne")
logging.info("Program rozpoczął pracę")
logging.warning("Brak części danych")

## Logger modułu

W większym projekcie zamiast funkcji na głównym loggerze często używamy:

```python
logger = logging.getLogger(__name__)
```

Dzięki temu wpis wiadomo, z którego modułu pochodzi.

In [ ]:
logger = logging.getLogger(__name__)

logger.info("Przetwarzanie pomiaru")

## Logowanie wyjątku z tracebackiem

W bloku `except`:

```python
logger.exception("Nie udało się przetworzyć pliku")
```

zapisuje komunikat oraz traceback aktualnego wyjątku.

## Przykład z W8: bezpieczniejsze wczytanie JSON

Połączmy pliki, JSON, wyjątki i logging.

In [ ]:
import json
from pathlib import Path


def load_json(path):
    path = Path(path)

    try:
        with path.open("r", encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        logger.error("Brak pliku: %s", path)
        raise
    except json.JSONDecodeError as error:
        logger.error(
            "Niepoprawny JSON w %s: %s",
            path,
            error,
        )
        raise

## Dlaczego ponownie `raise`?

Po:

```python
logger.error(...)
raise
```

rejestrujemy problem, ale nie udajemy, że operacja zakończyła się sukcesem.

Sam zapis do logu **nie jest obsługą błędu**, jeśli kod nie potrafi dalej poprawnie działać.

## Granice obsługi wyjątków

Dobrze zaprojektowany program często ma:

```text
niski poziom
↓
zgłasza precyzyjny wyjątek

warstwa domenowa
↓
ewentualnie tłumaczy wyjątek

punkt wejścia programu
↓
decyduje, jak poinformować użytkownika
```

Nie każdy poziom musi przechwytywać ten sam wyjątek.

## Przykład integrujący

```python
def main():
    try:
        data = load_json("data/config.json")
        run_application(data)
    except FileNotFoundError:
        print("Brak pliku konfiguracji")
        return 1
    except json.JSONDecodeError:
        print("Uszkodzony plik konfiguracji")
        return 2

    return 0
```

Punkt wejścia podejmuje decyzję, jak zakończyć program.

## Czego nie robić

Unikaj:

```python
except:
    pass
```

oraz bardzo szerokiego:

```python
try:
    # setki linii kodu
except Exception:
    ...
```

i używania wyjątków do zwykłego sterowania każdą iteracją, jeśli prosty warunek byłby czytelniejszy.

## Ćwiczenie

Napisz funkcję:

```python
load_measurements(path)
```

która:

1. odczytuje plik JSON,
2. zgłasza `FileNotFoundError`, jeśli pliku nie ma,
3. zamienia `JSONDecodeError` na własny `MeasurementFileError`,
4. zachowuje pierwotną przyczynę przez `raise ... from ...`,
5. loguje poprawne wczytanie na poziomie `INFO`,
6. w programie głównym wyświetla użytkownikowi krótki komunikat bez ukrywania nieoczekiwanych błędów.

## Podsumowanie

Po tym wykładzie powinieneś pamiętać:

- traceback jest podstawowym źródłem informacji o awarii,
- przechwytujemy możliwie konkretne wyjątki,
- `else` oddziela kod wykonywany po sukcesie,
- `finally` służy do kodu wykonywanego niezależnie od wyniku,
- `raise` pozwala świadomie zgłaszać niepoprawny stan,
- `raise ... from ...` zachowuje przyczynę wyjątku,
- bare `except:` zwykle jest złym wyborem,
- EAFP oznacza próbę operacji i obsługę oczekiwanej awarii,
- `breakpoint()` pozwala badać stan programu,
- `logging` skaluje diagnostykę lepiej niż przypadkowe `print()`.

## Następny etap kursu

Po W9 potrafimy już budować programy, które:

- przechowują dane,
- zapisują je do plików,
- reagują na błędy.

W10 i W11 rozwijają organizację programu przez OOP, a następnie przejdziemy do iteratorów, generatorów, comprehensions i dekoratorów.